In [0]:
raw_path = "/Volumes/main/default/airbnb_volume/"
checkpoint_path = "/Volumes/main/default/airbnb_volume/checkpoints"

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("price", DoubleType(), True)
])

In [0]:
df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "csv")
      .schema(schema)
      .load(raw_path))

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/airbnb_volume/city_prices.csv")

df.show()

In [0]:
df.write \
  .format("delta") \
  .mode("overwrite") \
  .saveAsTable("workspace.default.bronze_airbnb")

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
df = spark.readStream \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/airbnb_volume/")

In [0]:
static_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Volumes/workspace/default/airbnb_volume/city_prices.csv")

static_df.printSchema()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("price", IntegerType(), True)
])

In [0]:
df = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/airbnb_volume/")

In [0]:
df.writeStream \
  .format("delta") \
  .option("checkpointLocation", "/Volumes/workspace/default/airbnb_volume/checkpoints/bronze_airbnb") \
  .outputMode("append") \
  .trigger(availableNow=True) \
  .table("workspace.default.bronze_airbnb")

In [0]:
%sql
SELECT * FROM workspace.default.bronze_airbnb;

In [0]:
df = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/airbnb_volume/")

df.isStreaming

In [0]:
from pyspark.sql.functions import col, upper, current_timestamp

bronze_df = spark.read.table("workspace.default.bronze_airbnb")

silver_df = bronze_df \
    .filter(col("id").isNotNull()) \
    .withColumn("city", upper(col("city"))) \
    .withColumn("price", col("price").cast("int")) \
    .withColumn("ingestion_time", current_timestamp())

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_airbnb")

In [0]:
%sql
SELECT * FROM workspace.default.silver_airbnb;

In [0]:
from pyspark.sql.functions import avg, count

silver_df = spark.read.table("workspace.default.silver_airbnb")

gold_df = silver_df \
    .groupBy("city") \
    .agg(
        avg("price").alias("avg_price"),
        count("*").alias("total_listings")
    )

gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_airbnb")

In [0]:
%sql
SELECT * FROM workspace.default.gold_airbnb;

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, upper, current_timestamp, avg, count

# -------------------------
# Define Schema
# -------------------------
schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("city", StringType(), True),
    StructField("price", IntegerType(), True)
])

# =========================
# 1️⃣ Bronze Layer
# =========================
bronze_stream = spark.readStream \
    .schema(schema) \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/airbnb_volume/")

bronze_query = bronze_stream.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/airbnb_volume/checkpoints/bronze_airbnb") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("workspace.default.bronze_airbnb")

bronze_query.awaitTermination()

# =========================
# 2️⃣ Silver Layer
# =========================
silver_stream = spark.readStream.table("workspace.default.bronze_airbnb")

silver_transformed = silver_stream \
    .filter(col("id").isNotNull()) \
    .withColumn("city", upper(col("city"))) \
    .withColumn("price", col("price").cast("int")) \
    .withColumn("ingestion_time", current_timestamp())

silver_query = silver_transformed.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/airbnb_volume/checkpoints/silver_airbnb") \
    .outputMode("append") \
    .trigger(availableNow=True) \
    .table("workspace.default.silver_airbnb")

silver_query.awaitTermination()

# =========================
# 3️⃣ Gold Layer
# =========================
gold_stream = spark.readStream.table("workspace.default.silver_airbnb")

gold_agg = gold_stream \
    .groupBy("city") \
    .agg(
        avg("price").alias("avg_price"),
        count("*").alias("total_listings")
    )

gold_query = gold_agg.writeStream \
    .format("delta") \
    .option("checkpointLocation", "/Volumes/workspace/default/airbnb_volume/checkpoints/gold_airbnb") \
    .outputMode("complete") \
    .trigger(availableNow=True) \
    .table("workspace.default.gold_airbnb")

gold_query.awaitTermination()

In [0]:
%sql
SELECT * FROM workspace.default.bronze_airbnb;